In [1]:
def caesar(text, shift=3):
    result = []
    for c in text:
        if c.isalpha():
            shifted = chr((ord(c.lower()) - ord('a') + shift) % 26 + ord('a'))
            result.append(shifted)
        else:
            result.append(c)
    return result

def make_caesar_pairs(n=5000, shift=3):
    import random, string
    src, tgt = [], []
    for _ in range(n):
        length = random.randint(3, 12)
        word = [random.choice(string.ascii_lowercase) for _ in range(length)]
        src.append(word)
        tgt.append(caesar(word, shift))
    return src, tgt

In [2]:
import sys, torch
sys.path.insert(0, '.')

from functools import partial
from torch.utils.data import DataLoader

from dataset   import Vocabulary, TranslationDataset, collate_fn
from model     import Transformer
from train     import train
from inference import greedy_decode, beam_search

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {DEVICE}")

# ── 1. Generate reverse-string pairs ─────────────────────────────────────────

src_sents, tgt_sents = make_caesar_pairs(n=10000, shift=4)
# Quick sanity check
for s, t in zip(src_sents[:5], tgt_sents[:5]):
    print(f"  {s}  →  {t}")

Using: cuda
  ['p', 'u', 'i', 't', 'z', 'b', 'j', 'n', 'u', 'l']  →  ['t', 'y', 'm', 'x', 'd', 'f', 'n', 'r', 'y', 'p']
  ['l', 'f', 'v', 'n', 'e', 'j']  →  ['p', 'j', 'z', 'r', 'i', 'n']
  ['i', 'x', 'j', 'q', 'f', 'p', 'r']  →  ['m', 'b', 'n', 'u', 'j', 't', 'v']
  ['g', 'x', 'd']  →  ['k', 'b', 'h']
  ['v', 'r', 'u', 'r']  →  ['z', 'v', 'y', 'v']


In [3]:
# Characters are the tokens, so src and tgt share the same vocab
src_vocab = Vocabulary()
tgt_vocab = Vocabulary()
src_vocab.build(src_sents)
tgt_vocab.build(tgt_sents)

print(f"Vocab size: {len(src_vocab)}")   # should be ~30 (26 letters + 4 specials)

# 80/20 train-val split
split = int(len(src_sents) * 0.8)
train_ds = TranslationDataset(src_sents[:split],  tgt_sents[:split],  src_vocab, tgt_vocab)
val_ds   = TranslationDataset(src_sents[split:],  tgt_sents[split:],  src_vocab, tgt_vocab)

pad_collate = partial(collate_fn, pad_idx=Vocabulary.PAD)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=pad_collate)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, collate_fn=pad_collate)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

Vocab size: 30
Train samples: 8000 | Val samples: 2000


In [5]:
import torch, torch.nn as nn
from torch.optim import Adam
import math, time

# Re-init model fresh
model = Transformer(
    src_vocab_size = len(src_vocab),
    tgt_vocab_size = len(tgt_vocab),
    d_model    = 128,
    h          = 4,
    N          = 3,
    d_ff       = 256,
    dropout    = 0.1,
    pad_idx    = Vocabulary.PAD,
    tie_weights = True,
).to(DEVICE)

# ── Flat LR instead of the warmup schedule ───────────────────────────────────
optimizer = Adam(model.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

from train import LabelSmoothingLoss, make_teacher_forcing_pair
criterion = LabelSmoothingLoss(len(tgt_vocab), pad_idx=Vocabulary.PAD, smoothing=0.1)

# ── Training loop ─────────────────────────────────────────────────────────────
def run_epoch(loader, training=True):
    model.train(training)
    total_loss, total_toks = 0.0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            src      = batch["src"].to(DEVICE)
            tgt_full = batch["tgt"].to(DEVICE)
            tgt_in, tgt_out = make_teacher_forcing_pair(tgt_full, Vocabulary.SOS, Vocabulary.EOS)

            logits = model(src, tgt_in)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B*T, V), tgt_out.reshape(B*T))

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            non_pad     = (tgt_out != Vocabulary.PAD).sum().item()
            total_loss += loss.item() * non_pad
            total_toks += non_pad

    avg = total_loss / max(total_toks, 1)
    return avg, math.exp(min(avg, 100))

print(f"{'Epoch':>6}  {'Train Loss':>10}  {'Train PPL':>10}  {'Val Loss':>10}  {'Val PPL':>10}")
print("─" * 55)

for epoch in range(1, 31):
    tr_loss, tr_ppl = run_epoch(train_loader, training=True)
    vl_loss, vl_ppl = run_epoch(val_loader,   training=False)
    scheduler.step(vl_loss)
    print(f"{epoch:>6}  {tr_loss:>10.4f}  {tr_ppl:>10.2f}  {vl_loss:>10.4f}  {vl_ppl:>10.2f}")
    if vl_ppl < 0.85:
        print(f"\n  ✓ Converged at epoch {epoch}")
        break

 Epoch  Train Loss   Train PPL    Val Loss     Val PPL
───────────────────────────────────────────────────────
     1      2.3201       10.18      1.7883        5.98
     2      1.4725        4.36      0.3631        1.44
     3      0.5129        1.67      0.0904        1.09
     4      0.2215        1.25      0.0731        1.08
     5      0.1396        1.15      0.0668        1.07
     6      0.1037        1.11      0.0543        1.06
     7      0.0818        1.09      0.0527        1.05
     8      0.0699        1.07      0.0395        1.04
     9      0.0601        1.06      0.0337        1.03
    10      0.0517        1.05      0.0314        1.03
    11      0.0460        1.05      0.0303        1.03
    12      0.0425        1.04      0.0268        1.03
    13      0.0372        1.04      0.0225        1.02
    14      0.0321        1.03      0.0200        1.02
    15      0.0321        1.03      0.0186        1.02
    16      0.0260        1.03      0.0194        1.02
    17   

In [13]:
import torch, os

os.makedirs("saved_models", exist_ok=True)

model_config = {
    "src_vocab_size": len(src_vocab),
    "tgt_vocab_size": len(tgt_vocab),
    "d_model":    128,
    "h":          4,
    "N":          3,
    "d_ff":       256,
    "dropout":    0.1,
    "pad_idx":    0,
    "tie_weights": True,
}

torch.save({
    "model_state":   model.state_dict(),
    "model_config":  model_config,
    "src_token2idx": src_vocab.token2idx,
    "src_idx2token": src_vocab.idx2token,
    "tgt_token2idx": tgt_vocab.token2idx,
    "tgt_idx2token": tgt_vocab.idx2token,
}, "saved_models/reverse_model.pt")

print("Saved to saved_models/caesar_model.pt")

Saved to saved_models/caesar_model.pt


In [10]:
import sys, torch
sys.path.insert(0, '.')

from model     import Transformer
from inference import greedy_decode
from dataset   import Vocabulary

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint = torch.load("saved_models/caesar_model.pt", map_location=DEVICE)

# Rebuild vocab objects from plain dicts
src_vocab = Vocabulary()
src_vocab.token2idx = checkpoint["src_token2idx"]
src_vocab.idx2token = checkpoint["src_idx2token"]

tgt_vocab = Vocabulary()
tgt_vocab.token2idx = checkpoint["tgt_token2idx"]
tgt_vocab.idx2token = checkpoint["tgt_idx2token"]

# Rebuild model
model = Transformer(**checkpoint["model_config"]).to(DEVICE)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print("Model loaded successfully.")

# Running inference on test words
test_words = ["hello", "world", "abc", "transformer", "python", "abcde"]

print("\n─── Greedy decode ───────────────────────────────")
for word in test_words:
    src = torch.tensor(
        [src_vocab.encode(list(word))], dtype=torch.long, device=DEVICE
    )
    greedy_ids = greedy_decode(model, src, Vocabulary.SOS, Vocabulary.EOS, device=DEVICE)
    greedy_out = "".join(tgt_vocab.idx2token[i] for i in greedy_ids
                         if i not in {Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK})

    correct    = ''.join(caesar(word,4))
    status     = "✓" if greedy_out == correct else "✗"
    print(f"  {status}  {word:15s} → {greedy_out:15s}  (expected: {correct})")

print("\n─── Beam search (beam=4) ────────────────────────")
for word in test_words:
    src = torch.tensor(
        [src_vocab.encode(list(word))], dtype=torch.long, device=DEVICE
    )
    beam_ids = beam_search(model, src, Vocabulary.SOS, Vocabulary.EOS, beam_size=4, device=DEVICE)
    beam_out = "".join(tgt_vocab.idx2token[i] for i in beam_ids
                       if i not in {Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK})

    correct  = ''.join(caesar(word,4))
    status   = "✓" if beam_out == correct else "✗"
    print(f"  {status}  {word:15s} → {beam_out:15s}  (expected: {correct})")

Model loaded successfully.

─── Greedy decode ───────────────────────────────
  ✓  hello           → lipps            (expected: lipps)
  ✓  world           → asvph            (expected: asvph)
  ✓  abc             → efg              (expected: efg)
  ✓  transformer     → xverwjsvqiv      (expected: xverwjsvqiv)
  ✓  python          → tcxlsr           (expected: tcxlsr)
  ✓  abcde           → efghi            (expected: efghi)

─── Beam search (beam=4) ────────────────────────
  ✓  hello           → lipps            (expected: lipps)
  ✓  world           → asvph            (expected: asvph)
  ✓  abc             → efg              (expected: efg)
  ✓  transformer     → xverwjsvqiv      (expected: xverwjsvqiv)
  ✓  python          → tcxlsr           (expected: tcxlsr)
  ✓  abcde           → efghi            (expected: efghi)
